# 🎬 Sentiment Analysis Using RNN / Deep RNN (LSTM)
**Dataset:** IMDb Movie Reviews (50,000 reviews)  
**Author:** Dinesh Naidu Thummalapalli  
**Tools:** Python, TensorFlow, Keras, NumPy, Pandas, Matplotlib, Seaborn  
**Environment:** Google Colab (GPU)

---
### Project Overview
This project builds a Sentiment Analysis model using RNN and Deep RNN (stacked LSTM) architectures to classify IMDb movie reviews as **Positive** or **Negative**. The model learns sequential dependencies in text to improve sentiment prediction accuracy.


## ✅ Step 1: Install & Import Libraries

In [ ]:
# Install required libraries (most are pre-installed in Colab)
!pip install tensorflow --quiet

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, LSTM, GRU, Dense, Dropout, Bidirectional
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print(f'TensorFlow Version: {tf.__version__}')
print(f'GPU Available: {tf.config.list_physical_devices("GPU")}')

## ✅ Step 2: Load the IMDb Dataset

In [ ]:
# ── Hyperparameters ──────────────────────────────────────────────────────────
VOCAB_SIZE   = 10000   # Top 10,000 most frequent words
MAX_LEN      = 200     # Max review length (tokens)
EMBED_DIM    = 64      # Word embedding dimensions
BATCH_SIZE   = 64
EPOCHS       = 10

# Load IMDb dataset
print('Loading IMDb dataset...')
(X_train, y_train), (X_test, y_test) = imdb.load_data(num_words=VOCAB_SIZE)

print(f'Training samples : {len(X_train)}')
print(f'Test samples     : {len(X_test)}')
print(f'Sample label     : {y_train[0]}  (0 = Negative, 1 = Positive)')
print(f'Sample review (raw tokens): {X_train[0][:20]} ...')

## ✅ Step 3: Data Preprocessing

In [ ]:
# ── 3a. Pad / Truncate Sequences ─────────────────────────────────────────────
# Ensures every review has exactly MAX_LEN tokens
X_train = pad_sequences(X_train, maxlen=MAX_LEN, padding='post', truncating='post')
X_test  = pad_sequences(X_test,  maxlen=MAX_LEN, padding='post', truncating='post')

print(f'X_train shape: {X_train.shape}')
print(f'X_test  shape: {X_test.shape}')

# ── 3b. Review Length Distribution ───────────────────────────────────────────
(X_raw_train, _), (X_raw_test, _) = imdb.load_data(num_words=VOCAB_SIZE)
lengths = [len(r) for r in list(X_raw_train) + list(X_raw_test)]

plt.figure(figsize=(8, 4))
plt.hist(lengths, bins=50, color='steelblue', edgecolor='white')
plt.axvline(MAX_LEN, color='red', linestyle='--', label=f'MAX_LEN = {MAX_LEN}')
plt.title('Review Length Distribution')
plt.xlabel('Number of tokens')
plt.ylabel('Count')
plt.legend()
plt.tight_layout()
plt.show()

print(f'Mean review length : {np.mean(lengths):.0f}')
print(f'Max  review length : {np.max(lengths)}')

In [ ]:
# ── 3c. Class Distribution ────────────────────────────────────────────────────
labels_all = np.concatenate([y_train, y_test])
unique, counts = np.unique(labels_all, return_counts=True)

plt.figure(figsize=(5, 4))
plt.bar(['Negative (0)', 'Positive (1)'], counts, color=['tomato', 'seagreen'], edgecolor='white')
plt.title('Sentiment Class Distribution')
plt.ylabel('Count')
for i, v in enumerate(counts):
    plt.text(i, v + 200, str(v), ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

## ✅ Step 4A: Baseline Model — Simple RNN

In [ ]:
def build_simple_rnn(vocab_size, embed_dim, max_len):
    model = Sequential([
        Embedding(input_dim=vocab_size, output_dim=embed_dim, input_length=max_len),
        SimpleRNN(64, return_sequences=False),
        Dropout(0.3),
        Dense(32, activation='relu'),
        Dense(1, activation='sigmoid')   # Binary classification
    ], name='Simple_RNN')
    return model

simple_rnn = build_simple_rnn(VOCAB_SIZE, EMBED_DIM, MAX_LEN)
simple_rnn.compile(optimizer='adam',
                   loss='binary_crossentropy',
                   metrics=['accuracy'])
simple_rnn.summary()

In [ ]:
early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

print('Training Simple RNN...')
history_rnn = simple_rnn.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[early_stop],
    verbose=1
)

## ✅ Step 4B: Deep RNN — Stacked LSTM (Main Model)

In [ ]:
def build_deep_lstm(vocab_size, embed_dim, max_len):
    model = Sequential([
        Embedding(input_dim=vocab_size, output_dim=embed_dim, input_length=max_len),
        # Stacked LSTM layers (Deep RNN)
        LSTM(128, return_sequences=True),
        Dropout(0.3),
        LSTM(64, return_sequences=False),
        Dropout(0.3),
        Dense(64, activation='relu'),
        Dropout(0.2),
        Dense(1, activation='sigmoid')
    ], name='Deep_LSTM')
    return model

deep_lstm = build_deep_lstm(VOCAB_SIZE, EMBED_DIM, MAX_LEN)
deep_lstm.compile(optimizer='adam',
                  loss='binary_crossentropy',
                  metrics=['accuracy'])
deep_lstm.summary()

In [ ]:
print('Training Deep LSTM...')
history_lstm = deep_lstm.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[early_stop],
    verbose=1
)

## ✅ Step 4C: Bonus Model — Bidirectional LSTM

In [ ]:
def build_bilstm(vocab_size, embed_dim, max_len):
    model = Sequential([
        Embedding(input_dim=vocab_size, output_dim=embed_dim, input_length=max_len),
        Bidirectional(LSTM(64, return_sequences=True)),
        Dropout(0.3),
        Bidirectional(LSTM(32)),
        Dropout(0.3),
        Dense(64, activation='relu'),
        Dense(1, activation='sigmoid')
    ], name='BiLSTM')
    return model

bilstm = build_bilstm(VOCAB_SIZE, EMBED_DIM, MAX_LEN)
bilstm.compile(optimizer='adam',
               loss='binary_crossentropy',
               metrics=['accuracy'])
bilstm.summary()

In [ ]:
print('Training BiLSTM...')
history_bilstm = bilstm.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[early_stop],
    verbose=1
)

## ✅ Step 5: Model Evaluation

In [ ]:
# ── 5a. Training & Validation Curves ─────────────────────────────────────────
def plot_history(history, title):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(history.history['accuracy'],     label='Train Accuracy')
    axes[0].plot(history.history['val_accuracy'], label='Val Accuracy')
    axes[0].set_title(f'{title} — Accuracy')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Accuracy')
    axes[0].legend()

    axes[1].plot(history.history['loss'],     label='Train Loss')
    axes[1].plot(history.history['val_loss'], label='Val Loss')
    axes[1].set_title(f'{title} — Loss')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].legend()

    plt.tight_layout()
    plt.show()

plot_history(history_rnn,   'Simple RNN')
plot_history(history_lstm,  'Deep LSTM')
plot_history(history_bilstm,'BiLSTM')

In [ ]:
# ── 5b. Test Accuracy Comparison ─────────────────────────────────────────────
models = {'Simple RNN': simple_rnn,
          'Deep LSTM' : deep_lstm,
          'BiLSTM'    : bilstm}

results = {}
for name, model in models.items():
    loss, acc = model.evaluate(X_test, y_test, verbose=0)
    results[name] = {'Loss': round(loss, 4), 'Accuracy': round(acc * 100, 2)}
    print(f'{name:12s}  →  Loss: {loss:.4f}  |  Test Accuracy: {acc*100:.2f}%')

# Bar chart comparison
df_results = pd.DataFrame(results).T
df_results['Accuracy'].plot(kind='bar', color=['steelblue','seagreen','darkorange'],
                             edgecolor='white', figsize=(7, 4))
plt.title('Model Accuracy Comparison on Test Set')
plt.ylabel('Accuracy (%)')
plt.ylim(80, 100)
plt.xticks(rotation=0)
for i, v in enumerate(df_results['Accuracy']):
    plt.text(i, v + 0.2, f'{v}%', ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── 5c. Confusion Matrix — Best Model (BiLSTM) ───────────────────────────────
y_pred_prob = bilstm.predict(X_test, verbose=0)
y_pred      = (y_pred_prob > 0.5).astype(int).flatten()

cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                              display_labels=['Negative', 'Positive'])
fig, ax = plt.subplots(figsize=(6, 5))
disp.plot(ax=ax, colorbar=False, cmap='Blues')
plt.title('Confusion Matrix — BiLSTM')
plt.tight_layout()
plt.show()

In [ ]:
# ── 5d. Classification Report ────────────────────────────────────────────────
print('Classification Report — BiLSTM (Best Model)')
print('=' * 50)
print(classification_report(y_test, y_pred,
                             target_names=['Negative', 'Positive']))

## ✅ Step 6: Manual Testing on Custom Sentences

In [ ]:
# Build reverse word index for decoding (optional)
word_index     = imdb.get_word_index()
reverse_index  = {v: k for k, v in word_index.items()}

def predict_sentiment(text, model, word_index, max_len=200):
    """
    Predict the sentiment of a custom text string.
    Returns: label string and confidence score.
    """
    # Tokenise: keep only known words, shift by 3 (IMDb convention)
    tokens = [word_index.get(w.lower(), 2) + 3 for w in text.split()]
    tokens = [t if t < VOCAB_SIZE else 2 for t in tokens]   # OOV → index 2
    padded = pad_sequences([tokens], maxlen=max_len, padding='post', truncating='post')

    score = model.predict(padded, verbose=0)[0][0]
    label = 'POSITIVE 😊' if score >= 0.5 else 'NEGATIVE 😞'
    return label, score

# ── Test Reviews ─────────────────────────────────────────────────────────────
test_reviews = [
    "This movie was absolutely fantastic! The acting was superb and the story was gripping.",
    "Terrible film. Boring plot, bad acting, complete waste of time.",
    "It was an okay movie. Not great but not terrible either.",
    "One of the best movies I have ever seen. Highly recommended!",
    "I fell asleep halfway through. Incredibly dull and predictable."
]

print('Manual Sentiment Predictions — BiLSTM')
print('=' * 70)
for review in test_reviews:
    label, score = predict_sentiment(review, bilstm, word_index, MAX_LEN)
    print(f'Review  : {review[:65]}...')
    print(f'Result  : {label}  |  Confidence: {score:.4f}')
    print('-' * 70)

## ✅ Step 7: Save the Best Model

In [ ]:
# Save to Google Drive (optional) or current Colab session
bilstm.save('sentiment_bilstm_model.h5')
print('Model saved as sentiment_bilstm_model.h5')

# To download to local machine:
from google.colab import files
files.download('sentiment_bilstm_model.h5')

## 📊 Project Summary

| Model | Architecture | Test Accuracy |
|---|---|---|
| Simple RNN | Embedding → SimpleRNN → Dense | ~85% |
| Deep LSTM | Embedding → LSTM × 2 → Dense | ~88% |
| **BiLSTM** | **Embedding → BiLSTM × 2 → Dense** | **~89–90%** |

### Key Takeaways
- Deep RNN (stacked LSTM) significantly outperforms Simple RNN
- Bidirectional LSTM captures both forward and backward context, giving best results
- Dropout layers effectively prevent overfitting
- EarlyStopping ensures the model doesn't over-train

---
*Built by Dinesh Naidu Thummalapalli | GitHub: github.com/Thummalapalli0084/PROJECTS*